In [1]:
import pandas as pd
df = pd.read_parquet("../classify_data.parquet").reset_index()

In [2]:
class_map = {
    '0': 0,
    '1-3': 1,
    '4-7': 2,
    '8-14': 3,
    '15-30': 4,
    '31-60': 5
}
df['resolution_class_int'] = df['resolution_class'].map(class_map)

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

feature_columns = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip',
                    'Incident Address', 'City', 'Borough',
                    'Open Data Channel Type', 'Latitude', 'Longitude',
                    'hour_of_day', 'day_of_week', 'month',
                    'is_weekend', 'open_requests_same_day_zip']

X = df[feature_columns]
y = df['resolution_class_int']

#Segregate columsn for preprocessing
categorical_cols = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip',
                    'Incident Address', 'City', 'Borough',
                    'Open Data Channel Type',
                    'is_weekend']
numeric_cols = ['Latitude','Longitude','hour_of_day','day_of_week', 'month','open_requests_same_day_zip']

# Preprocessing -> Normalization + encoding categories
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

lr = LogisticRegression(
    solver='saga',      # MUCH faster for big sparse data
    max_iter=200,       # enough for saga
    tol=0.005,
    class_weight='balanced',
    n_jobs=8,          # use all CPU cores
    verbose=3           # show progress
)

# 4. Full pipeline: preprocessing + model
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', lr)
])

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro')

print(scores)

# 6. Fit model
model.fit(X_train, y_train)

# 7. Predict
y_pred = model.predict(X_test)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.18103722
Epoch 3, change: 0.11333646
Epoch 4, change: 0.082422472
Epoch 5, change: 0.07623235
Epoch 6, change: 0.049927421
Epoch 7, change: 0.044549981
Epoch 8, change: 0.039970241
Epoch 9, change: 0.038713885
Epoch 10, change: 0.037417614
Epoch 11, change: 0.035297099
Epoch 12, change: 0.034123287
Epoch 13, change: 0.034126398
Epoch 14, change: 0.031151344
Epoch 15, change: 0.028444855
Epoch 16, change: 0.02670131
Epoch 17, change: 0.026360978
Epoch 18, change: 0.024120934
Epoch 19, change: 0.021043948
Epoch 20, change: 0.021633015
Epoch 21, change: 0.021108324
Epoch 22, change: 0.019289989
Epoch 23, change: 0.016415504
Epoch 24, change: 0.015992481
Epoch 25, change: 0.015612579
Epoch 26, change: 0.015055214
Epoch 27, change: 0.014683677
Epoch 28, change: 0.014094865
Epoch 29, change: 0.013591418
Epoch 30, change: 0.013358063
Epoch 31, change: 0.013120868
Epoch 32, change: 0.012897578
Epoch 33, change: 0.012006121
Epoch 34, change: 0.011517503
Epo

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.16851527
Epoch 3, change: 0.12494933
Epoch 4, change: 0.07207263
Epoch 5, change: 0.066482505
Epoch 6, change: 0.067338599
Epoch 7, change: 0.047313507
Epoch 8, change: 0.042514505
Epoch 9, change: 0.036900571
Epoch 10, change: 0.040248192
Epoch 11, change: 0.035760489
Epoch 12, change: 0.031438505
Epoch 13, change: 0.031311804
Epoch 14, change: 0.025560833
Epoch 15, change: 0.026274378
Epoch 16, change: 0.024030834
Epoch 17, change: 0.023632825
Epoch 18, change: 0.023214623
Epoch 19, change: 0.022563366
Epoch 20, change: 0.022174035
Epoch 21, change: 0.021794027
Epoch 22, change: 0.021223087
Epoch 23, change: 0.020887276
Epoch 24, change: 0.020528796
Epoch 25, change: 0.019750239
Epoch 26, change: 0.019385803
Epoch 27, change: 0.018801094
Epoch 28, change: 0.018433069
Epoch 29, change: 0.017973579
Epoch 30, change: 0.017505019
Epoch 31, change: 0.016824236
Epoch 32, change: 0.016685246
Epoch 33, change: 0.015820178
Epoch 34, change: 0.015458872
Ep

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.16580879
Epoch 3, change: 0.12127919
Epoch 4, change: 0.080923023
Epoch 5, change: 0.067380869
Epoch 6, change: 0.0522595
Epoch 7, change: 0.040616836
Epoch 8, change: 0.039350844
Epoch 9, change: 0.036204338
Epoch 10, change: 0.034890166
Epoch 11, change: 0.033719298
Epoch 12, change: 0.033210694
Epoch 13, change: 0.034158381
Epoch 14, change: 0.028199977
Epoch 15, change: 0.026500424
Epoch 16, change: 0.02461527
Epoch 17, change: 0.02361768
Epoch 18, change: 0.022135574
Epoch 19, change: 0.021489971
Epoch 20, change: 0.02027835
Epoch 21, change: 0.016187344
Epoch 22, change: 0.016212549
Epoch 23, change: 0.015594566
Epoch 24, change: 0.015321961
Epoch 25, change: 0.014969024
Epoch 26, change: 0.013649163
Epoch 27, change: 0.013836849
Epoch 28, change: 0.0130546
Epoch 29, change: 0.012857366
Epoch 30, change: 0.012408555
Epoch 31, change: 0.012114339
Epoch 32, change: 0.011870244
Epoch 33, change: 0.01113656
Epoch 34, change: 0.011047372
Epoch 35,

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.16375784
Epoch 3, change: 0.11636858
Epoch 4, change: 0.1104866
Epoch 5, change: 0.065750292
Epoch 6, change: 0.052243265
Epoch 7, change: 0.064383105
Epoch 8, change: 0.061104934
Epoch 9, change: 0.036478169
Epoch 10, change: 0.032867309
Epoch 11, change: 0.033037729
Epoch 12, change: 0.031641915
Epoch 13, change: 0.031105572
Epoch 14, change: 0.027504991
Epoch 15, change: 0.02774726
Epoch 16, change: 0.026632749
Epoch 17, change: 0.023118018
Epoch 18, change: 0.021713759
Epoch 19, change: 0.021334855
Epoch 20, change: 0.020849067
Epoch 21, change: 0.020529011
Epoch 22, change: 0.020002567
Epoch 23, change: 0.019612823
Epoch 24, change: 0.019323189
Epoch 25, change: 0.018947331
Epoch 26, change: 0.018625308
Epoch 27, change: 0.017763235
Epoch 28, change: 0.017691727
Epoch 29, change: 0.016976417
Epoch 30, change: 0.0166603
Epoch 31, change: 0.016112287
Epoch 32, change: 0.015869654
Epoch 33, change: 0.015022751
Epoch 34, change: 0.014710016
Epoch 

In [10]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Test Accuracy: 0.7504745860729417

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.84      0.91    331345
           1       0.53      0.63      0.57     76042
           2       0.31      0.39      0.35     27143
           3       0.29      0.43      0.35     17086
           4       0.29      0.43      0.34     15160
           5       0.33      0.63      0.44     10482

    accuracy                           0.75    477258
   macro avg       0.46      0.56      0.49    477258
weighted avg       0.81      0.75      0.77    477258


Confusion Matrix:
 [[279487  34005   7094   3699   3611   3449]
 [  4967  47755  10710   5381   4266   2963]
 [   647   5837  10608   4706   3278   2067]
 [   284   1635   3021   7306   3048   1792]
 [   184    924   1807   2951   6446   2848]
 [    96    315    734   1007   1762   6568]]


In [8]:
import numpy as np
np.save("../results/regresssion_predictions.npy", y_pred)

In [9]:
y_pred = np.load("../results/regresssion_predictions.npy")

In [ ]:
print(X.shape)
print(y.shape)
df.dtypes